<a href="https://colab.research.google.com/github/LepingWan/ECE247A/blob/main/HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 1

**(a)** Let $h=Wx$ be the compressed representation and $\hat{x}=W^Th$ be the reconstruction. Minimizing
$$
L=\frac12\|\hat{x}-x\|^2
$$
makes the reconstruction close to the original input. For this to happen, $h$ must retain information about $x$, even though it has fewer coordinates. Information lost during encoding cannot be recovered by the decoder.

**(b)** Define
$$
h=Wx,\qquad \hat{x}=W^Th,\qquad e=\hat{x}-x,\qquad L=\frac12e^Te.
$$
The computational graph is:

```text
W ------------+
              v
x ---------> [Wx] -> h -----> [W^T h] -> x_hat -> [x_hat - x] -> e -> [e^T e / 2] -> L
                                ^                    ^
W -----> [transpose] -----------+                    |
x --------------------------------------------------+
```

Both occurrences of $W$ refer to the same matrix.

**(c)** Temporarily separate the two occurrences:
$$
F(A,B)=\frac12\|B^TAx-x\|^2,\qquad L(W)=F(W,W).
$$
By the chain rule,
$$
\frac{\partial L}{\partial W_{ij}}
=\left.\frac{\partial F}{\partial A_{ij}}\right|_{A=B=W}
+\left.\frac{\partial F}{\partial B_{ij}}\right|_{A=B=W}.
$$
Thus, the gradient contributions from the encoder and decoder paths must be added.

**(d)** Starting from the loss,
$$
\frac{\partial L}{\partial \hat{x}}=e.
$$
For the decoder, $\hat{x}_i=\sum_j W_{ji}h_j$, so its contribution is
$$
\left(\frac{\partial L}{\partial W_{ji}}\right)_{\mathrm{decoder}}=e_i h_j,
\qquad (\nabla_WL)_{\mathrm{decoder}}=he^T.
$$
The gradient reaching $h$ is
$$
\frac{\partial L}{\partial h}=We.
$$
Since $h=Wx$, the encoder contribution is
$$
(\nabla_WL)_{\mathrm{encoder}}=(We)x^T.
$$
Adding the two,
$$
\boxed{\nabla_WL=(Wx)(W^TWx-x)^T+W(W^TWx-x)x^T.}
$$
Both terms have shape $m\times n$, matching $W$.


# Problem 2

Write
$$
K=\alpha XX^T+\beta^{-1}I.
$$
Since $K$ is symmetric, $K^{-T}=K^{-1}$.

**(a)** The computational graph for $L_1$ is:

```text
X ----------------------+
|                       v
+-> [transpose] -> X^T -> [multiply] -> XX^T -> [times alpha] -> [+ beta^(-1) I] -> K
                                                                                 |
                                                                                 v
L1 <- [times (-D/2)] <- [log] <- [determinant] <------------------------------------+
```

**(b)** First differentiate with respect to $K$:
$$
\frac{\partial L_1}{\partial K}=-\frac D2K^{-T}=-\frac D2K^{-1}.
$$
To backpropagate through $K=\alpha XX^T+\beta^{-1}I$, let
$$
G=\frac{\partial L_1}{\partial K}.
$$
Since
$$
K_{ij}=\alpha\sum_k X_{ik}X_{jk}+\beta^{-1}\delta_{ij},
$$
each $X_{ab}$ contributes through both factors:
$$
\frac{\partial L_1}{\partial X_{ab}}
=\alpha\sum_jG_{aj}X_{jb}+\alpha\sum_iG_{ia}X_{ib}.
$$
In matrix form,
$$
\frac{\partial L_1}{\partial X}=\alpha(G+G^T)X.
$$
Here $G$ is symmetric, giving
$$
\boxed{\frac{\partial L_1}{\partial X}=-\alpha D K^{-1}X.}
$$

**(c)** The computational graph for $L_2$ is:

```text
X ----------------------+
|                       v
+-> [transpose] -> X^T -> [multiply] -> XX^T -> [times alpha] -> [+ beta^(-1) I] -> K
                                                                                 |
                                                                                 v
                                                                             [inverse]
                                                                                 |
                                                                                 v
Y ----------------------+                                                       K^(-1)
|                       v                                                        |
+-> [transpose] -> Y^T -> [multiply] -> YY^T --------------------------------+     |
                                                                           v     v
                                                                           [multiply]
                                                                                |
                                                                                v
                                                                             [trace]
                                                                                |
                                                                                v
                                                                          [times (-1/2)]
                                                                                |
                                                                                v
                                                                                L2
```

**(d)** Let $Q=K^{-1}$. Then
$$
L_2=-\frac12\operatorname{tr}(QYY^T).
$$
Using $\partial\operatorname{tr}(QA)/\partial Q=A^T$,
$$
\frac{\partial L_2}{\partial Q}=-\frac12(YY^T)^T=-\frac12YY^T.
$$
Using the inverse derivative given in the problem,
$$
\frac{\partial L_2}{\partial K}
=-K^{-1}\frac{\partial L_2}{\partial K^{-1}}K^{-1}
=\frac12K^{-1}YY^TK^{-1}.
$$
This matrix is symmetric. Applying the same chain rule through $XX^T$ as in part (b),
$$
\begin{aligned}
\frac{\partial L_2}{\partial X}
&=\alpha\left(\frac{\partial L_2}{\partial K}+\left(\frac{\partial L_2}{\partial K}\right)^T\right)X\\
&=\boxed{\alpha K^{-1}YY^TK^{-1}X}.
\end{aligned}
$$

**(e)** The constant $-c$ has zero derivative, so
$$
\begin{aligned}
\frac{\partial L}{\partial X}
&=\frac{\partial L_1}{\partial X}+\frac{\partial L_2}{\partial X}\\
&=\boxed{\alpha\left(K^{-1}YY^TK^{-1}-DK^{-1}\right)X}.
\end{aligned}
$$


# Problem 3

**(a)** Define
$$
z=W_1x+b_1,\qquad h=\sigma(z),\qquad u=W_2h+b_2,\qquad \hat{y}=u+x.
$$

```text
        W1        b1                       W2        b2
         v         v                        v         v
x ---> [W1 x] --> [+] -> z -> [sigmoid] -> h -> [W2 h] -> [+] -> u --> [+] -> y_hat
|                                                                    ^        |
+-------------------------- identity --------------------------------+        v
                                                                         [y_hat - y] <-- y
                                                                              |
                                                                              v
                                                                              e
                                                                              |
                                                                              v
                                                                         [e^T e / 2]
                                                                              |
                                                                              v
                                                                              L
```

**(b)** Let
$$
e=\hat{y}-y=W_2h+b_2+x-y.
$$
Then
$$
\frac{\partial L}{\partial\hat{y}}=e.
$$
The addition $\hat{y}=u+x$ passes the same gradient to $u$:
$$
\frac{\partial L}{\partial u}=e.
$$
Since $u_i=\sum_j(W_2)_{ij}h_j+(b_2)_i$,
$$
\frac{\partial L}{\partial(W_2)_{ij}}=e_i h_j.
$$
Thus,
$$
\boxed{\nabla_{W_2}L=eh^T.}
$$

**(c)** Since $\partial u_i/\partial(b_2)_j=\delta_{ij}$,
$$
\boxed{\nabla_{b_2}L=e=\hat{y}-y.}
$$
The residual connection changes the value of $\hat{y}$, and hence $e$, but the local derivative with respect to $b_2$ is still the identity.

**(d)** Backpropagating through $u=W_2h+b_2$,
$$
\frac{\partial L}{\partial h}=W_2^Te.
$$
For the sigmoid,
$$
\sigma'(z_i)=\sigma(z_i)(1-\sigma(z_i))=h_i(1-h_i).
$$
Applying the chain rule element by element,
$$
\frac{\partial L}{\partial z_i}=(W_2^Te)_i h_i(1-h_i).
$$
Let
$$
\delta=(W_2^Te)\odot h\odot(1-h),
$$
where $\odot$ means element-wise multiplication.

Since $z_i=\sum_j(W_1)_{ij}x_j+(b_1)_i$,
$$
\frac{\partial L}{\partial(W_1)_{ij}}=\delta_i x_j.
$$
Therefore,
$$
\boxed{\nabla_{W_1}L=\left[(W_2^Te)\odot h\odot(1-h)\right]x^T.}
$$
The skip connection affects this gradient through
$$
e=W_2h+b_2+x-y.
$$
There is no separate skip-path term for $W_1$, because the added $x$ does not depend on $W_1$.

**(e)** Since $\partial z_i/\partial(b_1)_j=\delta_{ij}$,
$$
\boxed{\nabla_{b_1}L=(W_2^Te)\odot h\odot(1-h).}
$$


## Two-layer neural network - Homework #3

**Neural Networks & Deep Learning - ECE C147A/C247A, UCLA Fall 2026**  
**Instructor:** Prof. Suhas Diggavi  
**TAs:** A. Kadiyala, B. Huang, M.Y. Turali, K. Pang, S.Y. Yetim  
**Released:** Monday, October 12, 2026  
**Due:** Friday, October 23, 2026, 11:59 PM Pacific Time  
**HW3 Problem 4:** 30 points

Please follow the notebook linearly to implement a two layer neural network.

Please print out the workbook entirely when completed.

We thank Serena Yeung & Justin Johnson for permission to use code written for the CS 231n class (cs231n.stanford.edu).  These are the functions in the cs231n folders and code in the jupyer notebook to preprocess and show the images.  The classifiers used are based off of code prepared for CS 231n as well.

The goal of this workbook is to give you experience with training a two layer neural network.

In [ ]:
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

%matplotlib inline
%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

## Toy example

Before loading CIFAR-10, there will be a toy example to test your implementation of the forward and backward pass

In [ ]:
from nndl.neural_net import TwoLayerNet

In [ ]:
# Create a small net and some toy data to check your implementations.
# Note that we set the random seed for repeatable experiments.

input_size = 4
hidden_size = 10
num_classes = 3
num_inputs = 5

def init_toy_model():
    np.random.seed(0)
    return TwoLayerNet(input_size, hidden_size, num_classes, std=1e-1)

def init_toy_data():
    np.random.seed(1)
    X = 10 * np.random.randn(num_inputs, input_size)
    y = np.array([0, 1, 2, 2, 1])
    return X, y

net = init_toy_model()
X, y = init_toy_data()

### Compute forward pass scores

In [ ]:
## Implement the forward pass of the neural network.

# Note, there is a statement if y is None: return scores, which is why 
# the following call will calculate the scores.
scores = net.loss(X)
print('Your scores:')
print(scores)
print()
print('correct scores:')
correct_scores = np.asarray([
    [-1.07260209,  0.05083871, -0.87253915],
    [-2.02778743, -0.10832494, -1.52641362],
    [-0.74225908,  0.15259725, -0.39578548],
    [-0.38172726,  0.10835902, -0.17328274],
    [-0.64417314, -0.18886813, -0.41106892]])
print(correct_scores)
print()

# The difference should be very small. We get < 1e-7
print('Difference between your scores and correct scores:')
print(np.sum(np.abs(scores - correct_scores)))

### Forward pass loss

In [ ]:
loss, _ = net.loss(X, y, reg=0.05)
correct_loss = 1.071696123862817

# should be very small, we get < 1e-12
print('Difference between your loss and correct loss:')
print(np.sum(np.abs(loss - correct_loss)))

In [ ]:
print(loss)

### Backward pass

Implements the backwards pass of the neural network.  Check your gradients with the gradient check utilities provided.

In [ ]:
from cs231n.gradient_check import eval_numerical_gradient

# Use numeric gradient checking to check your implementation of the backward pass.
# If your implementation is correct, the difference between the numeric and
# analytic gradients should be less than 1e-8 for each of W1, W2, b1, and b2.

loss, grads = net.loss(X, y, reg=0.05)

# these should all be less than 1e-8 or so
for param_name in grads:
    f = lambda W: net.loss(X, y, reg=0.05)[0]
    param_grad_num = eval_numerical_gradient(f, net.params[param_name], verbose=False)
    print('{} max relative error: {}'.format(param_name, rel_error(param_grad_num, grads[param_name])))

### Training the network

Implement neural_net.train() to train the network via stochastic gradient descent, much like the softmax and SVM.

In [ ]:
net = init_toy_model()
stats = net.train(X, y, X, y,
            learning_rate=1e-1, reg=5e-6,
            num_iters=100, verbose=False)

print('Final training loss: ', stats['loss_history'][-1])

# plot the loss history
plt.plot(stats['loss_history'])
plt.xlabel('iteration')
plt.ylabel('training loss')
plt.title('Training Loss history')
plt.show()

## Classify CIFAR-10

Do classification on the CIFAR-10 dataset.

In [ ]:
from cs231n.data_utils import load_CIFAR10

def get_CIFAR10_data(num_training=49000, num_validation=1000, num_test=1000):
    """
    Load the CIFAR-10 dataset from disk and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.  
    """
    # Load the raw CIFAR-10 data
    cifar10_dir = 'cifar-10-batches-py'
    X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)
        
    # Subsample the data
    mask = list(range(num_training, num_training + num_validation))
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = list(range(num_training))
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = list(range(num_test))
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean image
    mean_image = np.mean(X_train, axis=0)
    X_train -= mean_image
    X_val -= mean_image
    X_test -= mean_image

    # Reshape data to rows
    X_train = X_train.reshape(num_training, -1)
    X_val = X_val.reshape(num_validation, -1)
    X_test = X_test.reshape(num_test, -1)

    return X_train, y_train, X_val, y_val, X_test, y_test


# Invoke the above function to get our data.
X_train, y_train, X_val, y_val, X_test, y_test = get_CIFAR10_data()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

### Running SGD

If your implementation is correct, you should see a validation accuracy of around 28-29%.

In [ ]:
input_size = 32 * 32 * 3
hidden_size = 50
num_classes = 10
net = TwoLayerNet(input_size, hidden_size, num_classes)

# Train the network
stats = net.train(X_train, y_train, X_val, y_val,
            num_iters=1000, batch_size=200,
            learning_rate=1e-4, learning_rate_decay=0.95,
            reg=0.25, verbose=True)

# Predict on the validation set
val_acc = (net.predict(X_val) == y_val).mean()
print('Validation accuracy: ', val_acc)

# Save this net as the variable subopt_net for later comparison.
subopt_net = net

## Questions:

The training accuracy isn't great.  

(1) What are some of the reasons why this is the case?  Take the following cell to do some analyses and then report your answers in the cell following the one below.

(2) How should you fix the problems you identified in (1)?


In [ ]:
stats['train_acc_history']

In [ ]:
# ================================================================ #
# YOUR CODE HERE:
#   Do some debugging to gain some insight into why the optimization
#   isn't great.
# ================================================================ #

# Plot the loss function and train / validation accuracies

pass
# ================================================================ #
# END YOUR CODE HERE
# ================================================================ #

## Answers:

(1) You fill this in

(2) You fill this in

## Optimize the neural network

Use the following part of the Jupyter notebook to optimize your hyperparameters on the validation set.  Store your nets as best_net.

In [ ]:
best_net = None # store the best model into this 

# ================================================================ #
# YOUR CODE HERE:
#   Optimize over your hyperparameters to arrive at the best neural
#   network.  You should be able to get over 50% validation accuracy.
#   For this part of the notebook, we will give credit based on the
#   accuracy you get.  Your score on this question will be multiplied by:
#      min(floor((X - 28%)) / %22, 1) 
#   where if you get 50% or higher validation accuracy, you get full
#   points.
#
#   Note, you need to use the same network structure (keep hidden_size = 50)!
# ================================================================ #
pass

# ================================================================ #
# END YOUR CODE HERE
# ================================================================ #
best_net = net

In [ ]:
from cs231n.vis_utils import visualize_grid

# Visualize the weights of the network

def show_net_weights(net):
    W1 = net.params['W1']
    W1 = W1.T.reshape(32, 32, 3, -1).transpose(3, 0, 1, 2)
    plt.imshow(visualize_grid(W1, padding=3).astype('uint8'))
    plt.gca().axis('off')
    plt.show()

show_net_weights(subopt_net)
show_net_weights(best_net)

## Question:

(1) What differences do you see in the weights between the suboptimal net and the best net you arrived at?

## Answer:

(1) You fill this in

## Evaluate on test set 

In [ ]:
test_acc = (best_net.predict(X_test) == y_test).mean()
print('Test accuracy: ', test_acc)